In [1]:
# If running in a fresh environment, run this cell:
!pip uninstall -y tensorflow keras pennylane imbalanced-learn
!pip install tensorflow==2.15.0 keras==2.15.0 pennylane==0.39 imbalanced-learn

Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Successfully uninstalled tensorflow-2.17.1
Found existing installation: keras 3.5.0
Uninstalling keras-3.5.0:
  Successfully uninstalled keras-3.5.0
Found existing installation: imbalanced-learn 0.12.4
Uninstalling imbalanced-learn-0.12.4:
  Successfully uninstalled imbalanced-learn-0.12.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 3.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 43.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72

In [2]:
# Import required packages
import os
import shutil
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Quantum imports
import pennylane as qml

# Set random seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

In [3]:
# Define the base dataset path and class names
base_dataset_path = "/kaggle/input/oral-diseases"  # Adjust if needed
class_names = [
    "Calculus",
    "Data caries",
    "Gingivitis",
    "Mouth Ulcer",
    "ToothDiscoloration",
    "Hypodentia"
]
num_classes = len(class_names)

# Mapping from class name to folder path (adjust these as per your dataset)
cls_folder_map = {
    "Calculus": "Calculus/Calculus",
    "Data caries": "Data caries/Data caries/caries orignal data set/done",
    "Gingivitis": "Gingivitis/Gingivitis",
    "Mouth Ulcer": "Mouth Ulcer/Mouth Ulcer/ulcer original dataset/ulcer original dataset",
    "ToothDiscoloration": "Tooth Discoloration/Tooth Discoloration /tooth discoloration original dataset/tooth discoloration original dataset",
    "Hypodentia": "hypodontia/hypodontia"
}

# Copy all class folders into a local dataset directory
working_dir = "/kaggle/working/"
dataset_dir = os.path.join(working_dir, "dataset")
os.makedirs(dataset_dir, exist_ok=True)

for cls in class_names:
    src = os.path.join(base_dataset_path, cls_folder_map[cls])
    dst = os.path.join(dataset_dir, cls)
    shutil.copytree(src, dst, dirs_exist_ok=True)
print("All classes copied into local 'dataset' folder!")

# Create directories for train, validation, and test splits
train_dir = os.path.join(working_dir, "train")
val_dir   = os.path.join(working_dir, "val")
test_dir  = os.path.join(working_dir, "test")
for folder in [train_dir, val_dir, test_dir]:
    os.makedirs(folder, exist_ok=True)
    for cls in class_names:
        os.makedirs(os.path.join(folder, cls), exist_ok=True)

# Split each class into 70% train, 15% val, 15% test
for cls in class_names:
    class_path = os.path.join(dataset_dir, cls)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)
    n_total = len(images)
    n_train = int(0.7 * n_total)
    n_val   = int(0.15 * n_total)
    train_files = images[:n_train]
    val_files   = images[n_train:n_train + n_val]
    test_files  = images[n_train + n_val:]
    
    for f in train_files:
        shutil.move(os.path.join(class_path, f), os.path.join(train_dir, cls, f))
    for f in val_files:
        shutil.move(os.path.join(class_path, f), os.path.join(val_dir, cls, f))
    for f in test_files:
        shutil.move(os.path.join(class_path, f), os.path.join(test_dir, cls, f))
print("Train/Val/Test split completed!")

All classes copied into local 'dataset' folder!
Train/Val/Test split completed!


In [4]:
# Define image size and batch size for training
image_size = (224, 224)
batch_size = 32

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2
)
# Only rescaling for validation and testing
val_datagen   = ImageDataGenerator(rescale=1./255)
test_datagen  = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical"
)
val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical"
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

Found 3892 images belonging to 6 classes.
Found 831 images belonging to 6 classes.
Found 840 images belonging to 6 classes.


In [ ]:
# ===== Replacement Architectural Code =====

# --- Quantum Circuit based on PDF architecture (updated for batching) ---
num_qubits = 6
num_layers = 6  # Number of entanglement repetitions
dev = qml.device("default.qubit", wires=num_qubits)

@qml.qnode(dev, interface='tf', batching=True)
def quantum_circuit(inputs, weights):
    # 'inputs' is now expected to be batched with shape (batch_size, num_qubits)
    # Apply Hadamard gates to each qubit (automatically batched)
    for i in range(num_qubits):
        qml.Hadamard(wires=i)
    # Embed classical data using RY gates; use slicing to pick each qubit's data
    for i in range(num_qubits):
        qml.RY(inputs[:, i], wires=i)
    # Entanglement layers with parameterized rotations
    for layer in range(num_layers):
        # Apply RY rotations (first parameter)
        for i in range(num_qubits):
            qml.RY(weights[layer, i, 0], wires=i)
        # Entangle with CZ gates
        for i in range(num_qubits - 1):
            qml.CZ(wires=[i, i + 1])
        qml.CZ(wires=[num_qubits - 1, 0])
        # Apply RZ rotations (second parameter)
        for i in range(num_qubits):
            qml.RZ(weights[layer, i, 1], wires=i)
    # Final rotation and measurement preparation
    for i in range(num_qubits):
        qml.RX(weights[num_layers, i, 0], wires=i)
        qml.Hadamard(wires=i)
    # Return expectation values for each qubit; batching ensures shape (batch_size, num_qubits)
    return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

# Define weight shapes to accommodate 6 layers + final rotation row (using one parameter per final rotation)
weight_shapes = {"weights": (num_layers + 1, num_qubits, 2)}
quantum_layer = qml.qnn.KerasLayer(
    quantum_circuit, 
    weight_shapes, 
    output_dim=num_qubits,
    name="quantum_layer"
)

# --- Classical Backbone: Pretrained ResNet as feature extractor ---
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

# Load a pretrained ResNet50 model (without its top) to mimic ResNet-18 behavior
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(image_size[0], image_size[1], 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
# Reduce to features matching the number of qubits (using a tanh activation)
x = Dense(num_qubits, activation='tanh', name="pre_quantum_dense")(x)

# --- Hybrid Quantum-Classical Model ---
# Insert the quantum layer
x = quantum_layer(x)
# Final classification layer
outputs = Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()

/usr/local/lib/python3.10/dist-packages/pennylane/workflow/qnode.py:148: UserWarning: Received gradient_kwarg batching, which is not included in the list of standard qnode gradient kwargs.
  warnings.warn(


94765736/94765736 [==============================] - 0s 0us/step
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 230, 230, 3)          0         ['input_1[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 112, 112, 64)         9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 112, 112, 64)         256       ['conv1_conv[0][0]']          
 on)                         

In [6]:
# Define callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True, verbose=1)
]

# Train the model
history = model.fit(
    train_generator,
    epochs=50,  # adjust epochs as needed
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Epoch 1/50
122/122 [==============================] - ETA: 0s - loss: 1.3426 - accuracy: 0.6146
Epoch 1: val_loss improved from inf to 1.77190, saving model to best_model.h5


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


122/122 [==============================] - 233s 2s/step - loss: 1.3426 - accuracy: 0.6146 - val_loss: 1.7719 - val_accuracy: 0.2250 - lr: 1.0000e-04
Epoch 2/50
122/122 [==============================] - ETA: 0s - loss: 1.2103 - accuracy: 0.7197
Epoch 2: val_loss improved from 1.77190 to 1.65446, saving model to best_model.h5
122/122 [==============================] - 208s 2s/step - loss: 1.2103 - accuracy: 0.7197 - val_loss: 1.6545 - val_accuracy: 0.2250 - lr: 1.0000e-04
Epoch 3/50
122/122 [==============================] - ETA: 0s - loss: 1.1392 - accuracy: 0.7760
Epoch 3: val_loss did not improve from 1.65446
122/122 [==============================] - 206s 2s/step - loss: 1.1392 - accuracy: 0.7760 - val_loss: 1.6592 - val_accuracy: 0.1998 - lr: 1.0000e-04
Epoch 4/50
122/122 [==============================] - ETA: 0s - loss: 1.0827 - accuracy: 0.7955
Epoch 4: val_loss did not improve from 1.65446
122/122 [==============================] - 205s 2s/step - loss: 1.0827 - accuracy: 0.7955

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Reset the generator to ensure predictions align with the correct order
test_generator.reset()

# Generate predictions for the test set
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Get the class names from the generator
class_names = list(test_generator.class_indices.keys())

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title("Confusion Matrix")
plt.show()

# Print the classification report for detailed metrics
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

NameError: name 'test_generator' is not defined